In [ ]:
import sys
from pathlib import Path

# project root on sys.path so `horizon` and `eval` resolve from notebooks/
sys.path.insert(0, str(Path("..").resolve()))

import polars as pl

In [ ]:
# pick a dataset — only the CSV path needs to change
csv_path = Path("..") / "datasets" / "parking_150m" / "clean.csv"
parquet_path = csv_path.with_suffix(".parquet")

In [3]:
# stream csv → parquet without materialising. zstd compresses well on string-
# heavy data and is fast to decompress; row_group_size keeps row groups large
# enough that downstream column projections stay sequential on disk.
# Read every column as String: dirty input has int-looking columns that contain
# values like "45/46" or "N/A", and FD repair treats values categorically anyway
# — no point letting type inference fight the data.
header = pl.scan_csv(csv_path, n_rows=0).collect_schema().names()
schema = {col: pl.String for col in header}

pl.scan_csv(
    csv_path,
    schema_overrides=schema,
    null_values=["N/A", "NA", ""],
).sink_parquet(
    parquet_path,
    compression="zstd",
    row_group_size=1_000_000,
    statistics=True,
)

csv_mb = csv_path.stat().st_size / 1024 / 1024
pq_mb = parquet_path.stat().st_size / 1024 / 1024
print(f"{csv_path.name}: {csv_mb:,.1f} MB  →  {parquet_path.name}: {pq_mb:,.1f} MB")

clean.csv: 42,355.4 MB  →  clean.parquet: 3,453.4 MB


In [ ]:
# sanity check: row count + schema from the parquet
lf = pl.scan_parquet(parquet_path)
n_rows = lf.select(pl.len()).collect().item()
print(f"{n_rows:,} rows, {len(lf.collect_schema().names())} cols")
lf.head(5).collect()

145,483,454 rows, 19 cols


Plate,State,License Type,Summons Number,Issue Date,Violation Time,Violation,Judgment Entry Date,Fine Amount,Penalty Amount,Interest Amount,Reduction Amount,Payment Amount,Amount Due,Precinct,County,Issuing Agency,Violation Status,Summons Image
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""EMR6936""","""NY""","""PAS""","""4677744830""","""02/03/2020""","""12:24P""","""PHTO SCHOOL ZN SPEED VIOLATION""",null,"""$50,00""","""$0,00""","""$0,00""","""$0,00""","""$50,00""","""$0,00""","""000""","""QN""","""DEPARTMENT OF TRANSPORTATION""",null,"""View Summons (http://nycserv.n…"
"""GXY7539""","""NY""","""PAS""","""4677744956""","""02/03/2020""","""12:25P""","""PHTO SCHOOL ZN SPEED VIOLATION""",null,"""$50,00""","""$0,00""","""$0,00""","""$0,00""","""$50,00""","""$0,00""","""000""","""BK""","""DEPARTMENT OF TRANSPORTATION""",null,"""View Summons (http://nycserv.n…"
"""JHD4932""","""NY""","""PAS""","""8594264823""","""11/21/2019""","""11:20A""","""INSP. STICKER-EXPIRED/MISSING""",null,"""$65,00""","""$10,00""","""$0,00""","""$75,00""","""$0,00""","""$0,00""","""075""","""K""","""TRAFFIC""","""HEARING HELD-NOT GUILTY""","""View Summons (http://nycserv.n…"
"""GSW8206""","""PA""","""PAS""","""4677744981""","""02/03/2020""","""12:25P""","""PHTO SCHOOL ZN SPEED VIOLATION""",null,"""$50,00""","""$0,00""","""$0,00""","""$0,00""","""$50,00""","""$0,00""","""000""","""QN""","""DEPARTMENT OF TRANSPORTATION""",null,"""View Summons (http://nycserv.n…"
"""JHP6985""","""NY""","""PAS""","""4677744853""","""02/03/2020""","""12:24P""","""PHTO SCHOOL ZN SPEED VIOLATION""",null,"""$50,00""","""$0,00""","""$0,00""","""$0,00""","""$50,00""","""$0,00""","""000""","""BX""","""DEPARTMENT OF TRANSPORTATION""",null,"""View Summons (http://nycserv.n…"


: 

In [3]:
# sample 64M rows from parking_150m → parking_64m with a fixed seed. iterate
# the source parquet one row group at a time so we never hold the full 145M
# row index (or a 64M-element hash set) in memory — the previous `is_in`
# version blew up on exactly that.
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

src_parquet = Path("..") / "datasets" / "parking_150m" / "clean.parquet"
dst_parquet = Path("..") / "datasets" / "parking_64m" / "clean.parquet"
dst_parquet.parent.mkdir(parents=True, exist_ok=True)

pf = pq.ParquetFile(src_parquet)
n_total = pf.metadata.num_rows
rng = np.random.default_rng(42)
keep_idx = np.sort(rng.choice(n_total, size=64_000_000, replace=False))

writer = None
offset = 0
ptr = 0
for i in range(pf.num_row_groups):
    rg = pf.read_row_group(i)
    end = np.searchsorted(keep_idx, offset + rg.num_rows)
    local = keep_idx[ptr:end] - offset
    ptr = end
    sampled = rg.take(pa.array(local))
    if writer is None:
        writer = pq.ParquetWriter(
            dst_parquet, sampled.schema, compression="zstd"
        )
    writer.write_table(sampled)
    offset += rg.num_rows
writer.close()

pq_mb = dst_parquet.stat().st_size / 1024 / 1024
print(f"{dst_parquet.name}: {pq_mb:,.1f} MB")

clean.parquet: 1,612.3 MB


In [ ]:
# sanity check: row count + schema from the sampled parquet
lf = pl.scan_parquet(dst_parquet)
n_rows = lf.select(pl.len()).collect().item()
print(f"{n_rows:,} rows, {len(lf.collect_schema().names())} cols")
lf.head(5).collect()